# A08 · Entropy sweep on the crate stage

Coins collected, bombs planted and death rate per game during the short crate-stage runs of T05. There is one
line per entropy coefficient $c_H$, coloured on a log scale. All runs start from the same agent and differ only
in $c_H$.

**Reads** `runs/a2c_entropy_sweep/` (T05). **Writes** `figures/a2c_entropy_sweep.pdf`.

In [ ]:
import json
import pathlib
import shutil

import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import pandas as pd
import seaborn as sns

HERE = pathlib.Path.cwd().resolve()  # run from the notebook's folder or from the repository root
EXP = next(p / "experiments" for p in (HERE, *HERE.parents) if (p / "experiments" / "README.md").exists())
RUN = EXP / "runs" / "a2c_entropy_sweep"
FIGURES = EXP / "figures"
FIGURES.mkdir(exist_ok=True)

TEXTWIDTH = 16.0 / 2.54
sns.set_theme(context="paper", style="ticks", palette="colorblind", font="serif", rc={
    "text.usetex": shutil.which("latex") is not None,
    "text.latex.preamble": r"\usepackage[T1]{fontenc}\usepackage{amsmath}",
    "font.serif": ["cmr10", "Computer Modern Roman"], "mathtext.fontset": "cm", "axes.formatter.use_mathtext": True,
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9, "legend.fontsize": 8, "legend.title_fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.linewidth": 0.4, "grid.alpha": 0.6,
    "lines.linewidth": 1.1, "figure.dpi": 120, "figure.constrained_layout.use": True,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42,
})

## Data

In [ ]:
config = json.loads((RUN / "config.json").read_text())
history = pd.read_csv(RUN / "history.csv").query("stage == 'crates'").copy()
SMOOTH = 10  # rolling mean over iterations
for column in ("coins", "bombs", "death"):
    history[column] = history.groupby("entropy_coef")[column].transform(
        lambda s: s.rolling(SMOOTH, min_periods=1).mean())
coefs = sorted(history.entropy_coef.unique())
history.groupby("entropy_coef")[["coins", "bombs", "death"]].apply(lambda d: d.tail(50).mean()).round(3)

## Figure

In [ ]:
METRICS = {"coins": "coins per episode", "bombs": "bombs per episode", "death": "death rate"}
norm = colors.LogNorm(min(coefs), max(coefs))
palette = {c: plt.cm.viridis(norm(c)) for c in coefs}

fig, axes = plt.subplots(len(METRICS), 1, sharex=True, figsize=(TEXTWIDTH, 0.8 * TEXTWIDTH))
for ax, (column, label) in zip(axes, METRICS.items()):
    sns.lineplot(history, x="iteration", y=column, hue="entropy_coef", palette=palette, legend=False, lw=1.0, ax=ax)
    ax.set(ylabel=label, xlim=(0, history.iteration.max()))
axes[-1].set_xlabel("iteration of the crate stage")
fig.align_ylabels(axes)
axes[2].set_ylim(0, 1)
axes[0].set_ylim(bottom=0)
axes[1].set_ylim(bottom=0)
colorbar = fig.colorbar(plt.cm.ScalarMappable(cmap="viridis", norm=norm), ax=axes, pad=0.02, aspect=40)
colorbar.set_label(r"entropy coefficient $c_H$")
fig.savefig(FIGURES / "a2c_entropy_sweep.pdf")